# NB-08: WanModel End-to-End Forward Pass

## Learning Objectives
- See how noise, control, and reference latents are concatenated into the 48-channel input (D-08)
- Trace `WanModel.forward` from 48-channel video input through patchify, 3 DiT blocks (demo), Head, and unpatchify back to 16-channel output
- Understand gradient checkpointing: training wraps each block in `torch.utils.checkpoint.checkpoint()` to save VRAM; inference runs blocks directly (D-09)

## Prerequisites
- **Prior notebooks:** NB-06 (DiTBlock -- processes patchified tokens), NB-07 (patchify/unpatchify, Head -- the input/output projections)
- **Assumed concepts:** Diffusion model latent space, memory-compute tradeoffs in training

## Concept Map
- `WanModel` is the full assembled model -- every component it uses is a prior notebook's subject
- **48-channel input** = noise latent (16ch) + control latent (16ch) + reference latent (16ch) concatenated on `dim=1`
- **Gradient checkpointing** trades ~25% extra compute for lower peak VRAM during training -- requires `model.train()`

## WanModel Data Flow

```
WanModel.forward -- 8-Step Data Flow
=====================================
  Input: x [B, 48, F, H, W]  (48 = 16 noise + 16 ctrl + 16 ref)
         timestep [B]         (diffusion timestep, e.g. 500.0)
         context  [B, L, 4096] (T5 text embeddings)
         |
         +-- Step 1: time_embedding   [B] -> sinusoidal -> [B, 256] -> Linear+SiLU+Linear -> t [B, 1536]
         +-- Step 2: time_projection  t [B, 1536] -> SiLU+Linear -> [B, 9216] -> unflatten -> t_mod [B, 6, 1536]
         +-- Step 3: text_embedding   context [B, L, 4096] -> 2xLinear+GELU -> [B, L, 1536]
         +-- Step 4: patchify         x [B, 48, F, H, W] -> Conv3d -> [B, 1536, F, h, w] -> rearrange -> [B, S, 1536]
         +-- Step 5: freqs assembly   f_freqs+h_freqs+w_freqs cat -> [S, 1, 64]  (NB-06: full 3-band)
         +-- Step 6: N x DiTBlock    x [B, S, 1536] (shape unchanged through each block)  (NB-06)
         +-- Step 7: Head            x [B, S, 1536] -> adaLN + Linear -> [B, S, 64]  (NB-07)
         +-- Step 8: unpatchify      [B, S, 64] -> rearrange -> [B, 16, F, H, W]  (NB-07)
         |
  Output: x [B, 16, F, H, W]  (denoised output latents)
```

In [ ]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
_candidate = _here
for _ in range(10):  # search up to 10 levels
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
    _parent = _candidate.parent
    if _parent == _candidate:  # reached filesystem root
        break
    _candidate = _parent
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
_diffsynth_stub = types.ModuleType("diffsynth")
_models_stub = types.ModuleType("diffsynth.models")
sys.modules["diffsynth"] = _diffsynth_stub
sys.modules["diffsynth.models"] = _models_stub
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# NB-08 imports -- Source: diffsynth/models/wan_video_dit.py
from diffsynth.models.wan_video_dit import WanModel, sinusoidal_embedding_1d
import torch

print("Setup complete.")

## 1. The 48-Channel Input -- Noise, Control, and Reference Latents

The `WanModel` was designed for motion-controlled video generation. Its `in_dim=48` comes from concatenating three 16-channel latent tensors on the channel dimension:

| Component | Channels | Role |
|-----------|----------|------|
| `noise_latent` | 16 | The denoising target -- the noisy video frame the model is learning to denoise |
| `control_latent` | 16 | Control video (structure) -- the motion control signal (e.g., camera trajectory) |
| `ref_latent` | 16 | Reference image (appearance) -- the image whose style/content to preserve |

Each 16-channel component comes from a VAE encoder applied to video frames. The concatenation happens BEFORE the model is called, outside `WanModel.forward`. This lets the model's `patch_embedding` (a single Conv3d) jointly process all three conditioning signals from the first layer.

**Source:** `model_architecture.md` pipeline diagram; `diffsynth/models/wan_video_dit.py`, line 307 (`patch_embedding: Conv3d(in_dim=48, dim=1536, ...)`)

In [ ]:
# Source: model_architecture.md + diffsynth/models/wan_video_dit.py WanModel.forward
# D-08: show noise/control/ref as SEPARATE tensors before concatenation

B, F, H, W = 1, 4, 8, 8  # small spatial dims for CPU demo (seq_len = 4*4*4 = 64 after patchify)

noise_latent   = torch.randn(B, 16, F, H, W)   # [B, 16, F, H, W] -- denoising target
control_latent = torch.randn(B, 16, F, H, W)   # [B, 16, F, H, W] -- control video (structure)
ref_latent     = torch.randn(B, 16, F, H, W)   # [B, 16, F, H, W] -- reference image (appearance)

x_48 = torch.cat([noise_latent, control_latent, ref_latent], dim=1)  # [B, 48, F, H, W]
assert x_48.shape == torch.Size([B, 48, F, H, W]), \
    f"Expected ({B}, 48, {F}, {H}, {W}), got {x_48.shape}"

print(f"noise:    {noise_latent.shape}")
print(f"control:  {control_latent.shape}")
print(f"ref:      {ref_latent.shape}")
print(f"48-ch x:  {x_48.shape}  <- WanModel.in_dim=48")
print(f"\nChannels: {noise_latent.shape[1]} + {control_latent.shape[1]} + {ref_latent.shape[1]} = {x_48.shape[1]}")

## 2. WanModel Initialization

`WanModel.__init__` assembles the following sub-modules:

| Sub-module | Type | Purpose |
|-----------|------|---------|
| `patch_embedding` | `Conv3d(48, 1536, kernel=(1,2,2), stride=(1,2,2))` | Projects video to token sequence (NB-07) |
| `text_embedding` | `Sequential(Linear, GELU, Linear)` | Projects T5 text features from 4096-dim to model dim |
| `time_embedding` | `Sequential(Linear, SiLU, Linear)` | Projects sinusoidal timestep embedding (NB-01) to model dim |
| `time_projection` | `Sequential(SiLU, Linear(dim, dim*6))` | Produces the 6-parameter adaLN conditioning for DiT blocks (NB-05) |
| `blocks` | `ModuleList([DiTBlock(...) for _ in range(num_layers)])` | The N stacked DiT blocks (NB-06) |
| `head` | `Head(dim, out_dim, patch_size, eps)` | Output projection with 2-parameter adaLN (NB-07) |
| `freqs` | precomputed 3D RoPE frequency tensors | Used in freqs assembly inside forward (NB-06) |

The production WanModel uses `num_layers=30`. For this CPU demo we use `num_layers=3` to stay within STD-03's 5-second limit (verified: 0.042s on CPU with these dims).

**Source:** `diffsynth/models/wan_video_dit.py`, lines 274-328

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 273-328
# D-07: keep dim=1536 (real architecture dimensions), reduce layers to 3 for CPU demo

# --- DEMO CONFIG (3 layers, STD-03 compliant -- verified at 0.042s on CPU) ---
model = WanModel(
    dim=1536,           # production architecture dimension
    in_dim=48,          # 16 noise + 16 ctrl + 16 ref channels
    ffn_dim=8960,       # production FFN dimension
    out_dim=16,         # output: 16 latent channels
    text_dim=4096,      # T5 text encoder output dim
    freq_dim=256,       # sinusoidal timestep embedding dim
    eps=1e-6,
    patch_size=(1, 2, 2),
    num_heads=12,
    num_layers=3,       # REDUCED from 30 for CPU demo
    has_image_input=False,  # avoids needing clip_feature dummy tensor (RESEARCH.md Pitfall 3)
)

# --- PRODUCTION CONFIG (annotation only -- do not run on CPU) ---
# model = WanModel(dim=1536, in_dim=48, ffn_dim=8960, out_dim=16,
#     text_dim=4096, freq_dim=256, eps=1e-6, patch_size=(1, 2, 2),
#     num_heads=12, num_layers=30, has_image_input=True)
# Production WanModel has ~1.54B total parameters

total_params = sum(p.numel() for p in model.parameters())
print(f"Demo model (3 layers): {total_params:,} parameters")
print(f"Production model (30 layers, estimated): ~{total_params * 10:,} parameters")
print()
print("Sub-modules:")
for name, module in model.named_children():
    count = sum(p.numel() for p in module.parameters())
    print(f"  {name:<20}: {count:>12,} params")

### Production Configuration (30 Blocks)

The real Wan2.1-Fun-V1.1-1.3B-Control model uses 30 DiT blocks:

```python
WanModel(
    dim=1536, in_dim=48, ffn_dim=8960, out_dim=16,
    text_dim=4096, freq_dim=256, eps=1e-6,
    patch_size=(1, 2, 2), num_heads=12,
    num_layers=30,         # 30 stacked DiT blocks
    has_image_input=True,  # CLIP image encoder path active
)
```

With 30 blocks of ~51.2M params each, the full model has approximately **1.54 billion parameters**. This is why LoRA training is necessary -- fine-tuning all weights would require far too much memory. As computed in NB-06, the LoRA-targeted layers alone account for ~1.39B parameters across the 30 blocks.

This cell is annotation only -- do not run the 30-block config on CPU.

## 3. End-to-End Forward Pass

`WanModel.forward` executes an 8-step pipeline. Each step maps to a concept from a prior notebook:

| Step | Operation | Input Shape | Output Shape | Prior Notebook |
|------|-----------|-------------|--------------|---------------|
| 1 | `time_embedding` (sinusoidal + MLP) | `[B]` | `[B, 1536]` | NB-01 (sinusoidal embedding) |
| 2 | `time_projection` (SiLU + Linear + unflatten) | `[B, 1536]` | `[B, 6, 1536]` | NB-05 (adaLN-Zero 6-parameter) |
| 3 | `text_embedding` (2x Linear + GELU) | `[B, 20, 4096]` | `[B, 20, 1536]` | -- |
| 4 | `patchify` (Conv3d + rearrange) | `[B, 48, 4, 8, 8]` | `[B, 64, 1536]` | NB-07 (patchify) |
| 5 | freqs assembly (3-band cat) | -- | `[64, 1, 64]` | NB-06 (full 3-band freqs) |
| 6 | 3x `DiTBlock` | `[B, 64, 1536]` | `[B, 64, 1536]` | NB-06 (DiTBlock) |
| 7 | `Head` (adaLN + Linear) | `[B, 64, 1536]` | `[B, 64, 64]` | NB-07 (Head, t vs t_mod) |
| 8 | `unpatchify` (rearrange) | `[B, 64, 64]` | `[B, 16, 4, 8, 8]` | NB-07 (unpatchify) |

Note: `S = F * h * w = 4 * 4 * 4 = 64` tokens after patchify with stride `(1,2,2)` on `(F=4, H=8, W=8)` input.

**Source:** `diffsynth/models/wan_video_dit.py`, lines 359-416

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 359-416 (WanModel.forward)

model.eval()
timestep = torch.tensor([500.0])       # [B]           -- diffusion timestep
context  = torch.randn(B, 20, 4096)   # [B, L, 4096]  -- T5 text embeddings

# Shape trace through WanModel.forward (wan_video_dit.py lines 369-416):
# 1. time_embedding:   timestep [B] -> sinusoidal_embedding_1d(freq_dim=256, timestep)
#                      -> [B, 256] -> Linear+SiLU+Linear -> t [B, 1536]
# 2. time_projection:  t [B, 1536] -> SiLU+Linear -> [B, 9216] -> unflatten(1,(6,1536)) -> t_mod [B, 6, 1536]
# 3. text_embedding:   context [B, 20, 4096] -> Linear+GELU+Linear -> [B, 20, 1536]
# 4. patchify:         x_48 [B, 48, 4, 8, 8] -> Conv3d(stride=1,2,2) -> [B, 1536, 4, 4, 4]
#                      -> rearrange 'b c f h w -> b (f h w) c' -> [B, 64, 1536]
# 5. freqs assembly:   freqs[0][:4]+freqs[1][:4]+freqs[2][:4] cat on last dim -> [64, 1, 64]
#                      (64 = 22+21+21 = head_dim//2 = 128//2)
# 6. x3 DiTBlocks:     x [B, 64, 1536] (shape unchanged; each block applies adaLN+attn+ffn)
# 7. head:             x [B, 64, 1536] -> Head.forward(x, t) -> [B, 64, 64]  (64=16*1*2*2)
# 8. unpatchify:       [B, 64, 64] -> rearrange -> [B, 16, 4, 8, 8]

with torch.no_grad():
    out = model(x_48, timestep, context)   # [B, 16, 4, 8, 8]

assert out.shape == torch.Size([B, 16, F, H, W]), \
    f"Expected ({B}, 16, {F}, {H}, {W}), got {out.shape}"

print(f"Input:  {x_48.shape}   -- 48-channel concatenated latents")
print(f"Output: {out.shape}  -- 16-channel denoised output latents")
print(f"\nShape preserved: spatial (F={F}, H={H}, W={W}) unchanged; channels 48 -> 16")

### Shape Trace Summary

The complete shape transformation table for the demo config `(B=1, F=4, H=8, W=8, num_layers=3)`:

| Step | Operation | Input Shape | Output Shape |
|------|-----------|-------------|--------------|
| 1 | `time_embedding` | `[1]` | `[1, 1536]` |
| 2 | `time_projection` | `[1, 1536]` | `[1, 6, 1536]` |
| 3 | `text_embedding` | `[1, 20, 4096]` | `[1, 20, 1536]` |
| 4 | `patchify` | `[1, 48, 4, 8, 8]` | `[1, 64, 1536]` |
| 5 | freqs assembly | -- | `[64, 1, 64]` |
| 6 | 3x `DiTBlock` | `[1, 64, 1536]` | `[1, 64, 1536]` |
| 7 | `Head` | `[1, 64, 1536]` | `[1, 64, 64]` |
| 8 | `unpatchify` | `[1, 64, 64]` | `[1, 16, 4, 8, 8]` |

**Key observations:**
- After patchify (Step 4), `seq_len = F * (H//2) * (W//2) = 4 * 4 * 4 = 64` tokens
- Steps 6 (DiT blocks) preserve shape -- they transform features, not layout (NB-06)
- Step 7 (Head) projects from `dim=1536` to `out_dim * prod(patch_size) = 16 * 4 = 64` (NB-07)
- Step 8 (unpatchify) recovers video spatial layout from the flat token sequence (NB-07)

## 4. Gradient Checkpointing -- Training vs Inference

Gradient checkpointing is a memory-compute tradeoff for training large models. During the backward pass, PyTorch normally stores all intermediate activations from the forward pass to compute gradients. With 30 DiT blocks each holding ~51M parameters' worth of activations, peak VRAM becomes prohibitive.

**The checkpointing tradeoff:**
- **Without checkpointing (inference):** All block activations are stored. Fast, but high peak VRAM.
- **With checkpointing (training):** Only block inputs/outputs are stored. During backward, the forward pass is rerun for each block to recompute activations. Costs ~25% extra compute, but significantly reduces peak VRAM.

**The condition in `WanModel.forward` (line 393-412):**

```python
if self.training and use_gradient_checkpointing:
    x = torch.utils.checkpoint.checkpoint(
        create_custom_forward(block),
        x, context, t_mod, freqs,
        use_reentrant=True,
    )
else:
    x = block(x, context, t_mod, freqs)
```

**Critical:** The condition is `self.training AND use_gradient_checkpointing`. This means:
- `model.eval()` + `use_gradient_checkpointing=True` -> checkpoint path **never** runs (eval mode suppresses it)
- `model.train()` + `use_gradient_checkpointing=False` -> checkpoint path **never** runs (flag is False)
- `model.train()` + `use_gradient_checkpointing=True` -> checkpoint path **active**

This is Pitfall 4 from RESEARCH.md -- calling the model in eval mode with the flag set silently runs the direct path. No error, no warning.

**Source:** `diffsynth/models/wan_video_dit.py`, lines 393-412

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 393-412
# D-09: training vs inference side-by-side

# TRAINING mode -- gradient checkpointing active
# IMPORTANT: model.train() MUST be called BEFORE use_gradient_checkpointing=True
# Source: wan_video_dit.py line 393: `if self.training and use_gradient_checkpointing:`
model.train()
x_train = x_48.clone()  # fresh copy -- model.train() mode will set requires_grad on x internally
out_gc = model(x_train, timestep, context, use_gradient_checkpointing=True)
# Internally for each block:
#   x = x.requires_grad_(True)  # line 394 -- enables gradient tracking on x for use_reentrant=True
#   x = torch.utils.checkpoint.checkpoint(
#       create_custom_forward(block), x, context, t_mod, freqs, use_reentrant=True
#   )
# -> recomputes block activations during backward instead of storing them -> lower peak VRAM

# INFERENCE mode -- no checkpointing
model.eval()
with torch.no_grad():
    out_eval = model(x_48, timestep, context)
    # Internally: x = block(x, context, t_mod, freqs)  -- direct call, no recomputation

print("Training with checkpointing: saves VRAM by recomputing block activations on backward")
print("Inference without checkpointing: faster, no gradient tracking needed")
assert out_gc.shape == out_eval.shape, \
    f"Shape mismatch: {out_gc.shape} vs {out_eval.shape}"
print(f"Both paths produce shape: {out_gc.shape}")

### Why Gradient Checkpointing Matters for LoRA Training

In NB-06 we saw that a single DiTBlock with `dim=1536` has ~51.2M parameters. During training, the backward pass needs to store the **activations** at each block's input and output -- not just the parameters. For a batch of video frames at full resolution, the activation tensors can be 10-100x larger than the parameters themselves.

For the production 30-block WanModel:
- Each block processes sequences of `[B, seq_len, 1536]` -- at typical video resolutions (e.g., 512x512, 16 frames), `seq_len = 16 * 128 * 128 = 262,144` tokens
- Storing all 30 blocks' activations at that scale would require tens of GB just for activations

**Gradient checkpointing solution:** Store only the input to each block. During backward, rerun each block's forward pass to recompute its internal activations on demand. This reduces peak activation memory from O(num_layers) to O(1) layers at a time, at the cost of running each block's forward pass twice.

**Effect on LoRA training:** LoRA adds adapters to the attention and FFN matrices (NB-06 Section 3). The adapter parameters are small, but the activations flowing through the large frozen matrices are the memory bottleneck. Checkpointing makes LoRA training feasible on consumer GPUs.

In [ ]:
# From WanModel.forward, lines 393-412:
# The key branching logic inside the block loop:
#
# if self.training and use_gradient_checkpointing:
#     x = x.requires_grad_(True)  # enable grad tracking for use_reentrant=True
#
# for block in self.blocks:
#     if self.training and use_gradient_checkpointing:
#         # Training path: wrap block call in checkpoint
#         x = torch.utils.checkpoint.checkpoint(
#             create_custom_forward(block), x, context, t_mod, freqs, use_reentrant=True
#         )
#     else:
#         # Inference path: direct call
#         x = block(x, context, t_mod, freqs)
#
# Key insight: self.training is a PyTorch Module property set recursively.
# model.train() sets self.training=True for all submodules.
# model.eval() sets self.training=False for all submodules.

print("self.training controls the checkpoint branch:")
model.train()
print(f"  model.train() -> model.training = {model.training}")
print(f"  model.train() -> model.blocks[0].training = {model.blocks[0].training}")
model.eval()
print(f"  model.eval()  -> model.training = {model.training}")
print(f"  model.eval()  -> model.blocks[0].training = {model.blocks[0].training}")
print()
print("Pitfall: model.eval() + use_gradient_checkpointing=True silently runs the DIRECT path.")
print("The condition `self.training and use_gradient_checkpointing` is False when in eval mode.")

## Summary

### Key Takeaways
- **48-channel input composition**: `noise_latent (16ch)` + `control_latent (16ch)` + `reference_latent (16ch)` concatenated on `dim=1` before entering `WanModel`. Each 16-channel component comes from a VAE encoder. The `patch_embedding` Conv3d jointly processes all three conditioning signals.
- **WanModel.forward data flow**: sinusoidal timestep embedding (NB-01) -> time projection producing 6-parameter adaLN (NB-05) + text embedding -> patchify to token sequence (NB-07) -> freqs assembly for 3D RoPE (NB-06) -> N x DiTBlock (NB-06) -> Head (NB-07) -> unpatchify (NB-07). Every sub-step is a prior notebook's subject.
- **Gradient checkpointing**: Requires `model.train()` -- the condition `if self.training and use_gradient_checkpointing` means `model.eval()` always runs the direct block path regardless of the flag. Checkpointing trades ~25% extra compute for significant VRAM reduction by recomputing block activations during the backward pass instead of storing them.

### Source References
| Symbol | Location |
|--------|---------|
| `WanModel.__init__` | `diffsynth/models/wan_video_dit.py`, line 274 |
| `WanModel.patchify` | `diffsynth/models/wan_video_dit.py`, line 340 |
| `WanModel.unpatchify` | `diffsynth/models/wan_video_dit.py`, line 352 |
| `WanModel.forward` | `diffsynth/models/wan_video_dit.py`, line 359 |
| freqs assembly | `diffsynth/models/wan_video_dit.py`, lines 381-385 |
| checkpoint branch | `diffsynth/models/wan_video_dit.py`, lines 393-412 |

## Exercises

### Exercise 1 -- Minimum functional WanModel
Change `num_layers=1` when creating `WanModel`. Run the forward pass with the same `x_48`, `timestep`, and `context`. Does the output shape change? (It should not -- shape is determined by architecture dims and input spatial dims, not block count.) Compare the parameter count between 1-layer and 3-layer models. How many parameters does a single DiTBlock add?

### Exercise 2 -- Inspect the time embedding
Extract the intermediate time embedding by manually running the embedding: `t = model.time_embedding(sinusoidal_embedding_1d(256, timestep).to(torch.float32))`. Print `t.shape`, `t.mean().item()`, and `t.std().item()`. Then extract `t_mod = model.time_projection(t).unflatten(1, (6, 1536))`. How does the SiLU nonlinearity in `time_embedding` (Linear -> **SiLU** -> Linear) change the statistics compared to a plain linear transform?

### Exercise 3 -- Scale to larger spatial dimensions
Change the spatial dims to `F=8, H=16, W=16` (seq_len = 8*8*8 = 512 tokens after patchify). Update `x_48 = torch.cat([torch.randn(B, 16, F, H, W)] * 3, dim=1)` and `context = torch.randn(B, 20, 4096)`. Run the forward pass and verify the output shape is `[1, 16, 8, 16, 16]`. Time it with `import time; t0 = time.time(); model(...); print(time.time()-t0)`. How does the time scale compared to `seq_len=64`?